In [ ]:
import json
from pprint import pp
from time import sleep

import requests

## Set up API endpoints

In [ ]:
DS_HOSTNAME = "http://localhost:8000"
CUST_HOSTNAME = "http://localhost:9000"

## Populate a dataset

In [ ]:
# Example of one DPO data sample
with open("data0/train_example_data.jsonl", "r") as f:
    for i in range(55):
        line = json.loads(f.readline())
    pp(line, indent=2, width=140)

In [ ]:
# Create dataset in Datastore
data = {"name": "example-dpo-dataset", "description": "temp dataset for dpo demo"}

resp = requests.post(f"{DS_HOSTNAME}/v1/datasets", json=data)
pp(resp.json())
dataset_id = resp.json()["id"]

In [ ]:
# Upload dataset files
train_file = "data0/train_example_data.jsonl"
files = {"file": open(train_file, "rb")}
r = requests.post(
    f"{DS_HOSTNAME}/v1/datasets/{dataset_id}/files/contents/training/train_example_data.jsonl",
    files=files,
)
pp(r.json())

validation_file = "data0/val_example_data.jsonl"
files = {"file": open(validation_file, "rb")}
r = requests.post(
    f"{DS_HOSTNAME}/v1/datasets/{dataset_id}/files/contents/validation/val_example_data.jsonl",
    files=files,
)
pp(r.json())

test_file = "data0/test_example_data.jsonl"
files = {"file": open(validation_file, "rb")}
r = requests.post(
    f"{DS_HOSTNAME}/v1/datasets/{dataset_id}/files/contents/testing/test_example_data.jsonl",
    files=files,
)
pp(r.json())

## Customize meta/llama3-8b-instruct with LoRA DPO

In [ ]:
data = {
    "parent_model_id": "meta/llama3-8b-instruct",
    "dataset": dataset_id,
    "sha": "main",
    "hyperparameters": {
        "training_type": "lora-dpo",
        "epochs": 1,
        "batch_size": 8,
        "learning_rate": 0.0001,
        "ref_policy_kl_penalty": 0.1,
    },
}

resp = requests.post(f"{CUST_HOSTNAME}/v2/customizations", json=data)
pp(resp.json())
cust_id = resp.json()["id"]

### Check training progress

In [ ]:
resp = requests.get(f"{CUST_HOSTNAME}/v2/customizations/{cust_id}")
pp(resp.json())

In [ ]:
model_dir = resp.json()["model_id"]
status = resp.json()["status"]

while status != "TRAINING_COMPLETED":
    sleep(30)
    resp = requests.get(f"{CUST_HOSTNAME}/v2/customizations/{cust_id}")
    status = resp.json()["status"]
pp(resp.json())

### Get metrics

In [ ]:
# show metrics in MLFlow window

In [ ]:
# resp = requests.get(f"{CUST_HOSTNAME}/v2/customizations/{cust_id}/metrics")
# pp(resp.json())

## Show model checkpoint uploaded in datastore

In [ ]:
r = requests.get(f"{DS_HOSTNAME}/v1/models/{model_dir}")
pp(r.json())

## Inference on aligned checkpoint

In [ ]:
NIM_HOSTNAME = "http://localhost:9001"

In [ ]:
with open("data0/test_example_data.jsonl", "r") as f:
    for i in range(35):
        line = json.loads(f.readline())
        prompt = line["prompt"]
print(prompt)

In [ ]:
data = {
    "model": f"meta-llama3-8b-instruct-{dataset_id}-lora-dpo",
    "prompt": prompt,
    "max_tokens": 100,
    "seed": 1,
    "temperature": 0.1,
    "top_p": 1,
    "n": 1,
    "stop": "\n",
    "stream": False,
    "frequency_penalty": 0.0,
}
print(data)
r = requests.post(f"{NIM_HOSTNAME}/v1/completions", json=data)
pp(r.json())

## Cleanup/reset

In [ ]:
# delete dataset
r = requests.delete(f"{DS_HOSTNAME}/v1/datasets/{dataset_id}")
r.json()

In [ ]:
# delete model
model_id = resp.json()["model_id"]
r = requests.delete(f"{DS_HOSTNAME}/v1/models/{model_id}")
r.json()